# Clase 3 : Modelos Lineales Regularizados

En esta sesión, aprenderemos como, haciendo uso de sklearn, estimar modelos de regresión lineal regulariazados (e.g Lasso, Ridge, ElasticNet). Adicional a esto, declararemos una serie de funciones que serán de utilidad en la generación de rezagos en la data (**crealags**), para generar listas ordenadas con todos los modelos que queramos estimar (**creamodelos**),  y finalmente otra que será útil para la presentación de las métricas de desempeño de dichos modelos en cada una de las muestras (**metricas**).

Primero cargaremos la base de datoa a utilizar.

In [ ]:
# Instalar dependencias (ejecutar una vez si da error)
%pip install openpyxl statsmodels -q

In [ ]:
import pandas as pd
import subprocess
import sys

# Instalar openpyxl si no está instalado (necesario para .xlsx)
try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'])

excel_file = 'data_usd-clp.xlsx'
data = pd.read_excel(excel_file, engine='openpyxl')

## Descripción de la base de datos

La base **data_usd-clp.xlsx** contiene series temporales de variables financieras y macroeconómicas relacionadas con el tipo de cambio USD/CLP (peso chileno).

| Variable | Descripción |
|----------|-------------|
| **USDCLP Curncy** | Tipo de cambio USD/CLP spot (variable objetivo) |
| **USSW1 Curncy** | Tasa swap USD 1 año |
| **USSW10 Curncy** | Tasa swap USD 10 años |
| **CHSWP1 ICCH Curncy** | Swap Chile 1 año |
| **CHSWP10 Curncy** | Swap Chile 10 años |
| **CCHIL1U5 CBIN Curncy** | Credit default swap Chile 5 años |
| **HGA Comdty** | Oro (commodity) |
| **CL1 Comdty** | Petróleo WTI |
| **VIX Index** | Índice de volatilidad implícita |

In [ ]:
# Dimensiones y estructura
print("Dimensiones:", data.shape)
print("\nInformación de columnas:")
data.info()
print("\n" + "="*60)
print("Primeras filas:")
data.head(10)

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas de las variables numéricas:")
data.describe().round(4)

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas de las variables numéricas:")
data.describe().round(4)

## Análisis Exploratorio: Gráficos y Descriptivas

Antes de la estimación de modelos, realizamos un análisis exploratorio visual de la distribución de las variables mediante histogramas, gráficos de densidad (KDE) y sus combinaciones.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Seleccionar solo columnas numéricas
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
print("Variables numéricas:", numeric_cols)

In [ ]:
# Matriz de correlación (heatmap)
corr_matrix = data[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Matriz de correlación entre variables')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Histogramas individuales (una variable por subplot)
n_vars = len(numeric_cols)
n_cols_grid = 3
n_rows = (n_vars + n_cols_grid - 1) // n_cols_grid
fig, axes = plt.subplots(n_rows, n_cols_grid, figsize=(14, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(data[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Histogramas de las variables numéricas', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Gráficos de densidad (KDE) - una por variable
n_vars = len(numeric_cols)
n_rows = (n_vars + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(14, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()
for i, col in enumerate(numeric_cols):
    data[col].dropna().plot(kind='kde', ax=axes[i])
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Gráficos de densidad (KDE) de las variables', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Histograma + densidad superpuesta (combinación)
n_vars = len(numeric_cols)
n_rows = (n_vars + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(14, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()
for i, col in enumerate(numeric_cols):
    data[col].dropna().hist(bins=30, density=True, alpha=0.6, edgecolor='black', ax=axes[i])
    data[col].dropna().plot(kind='kde', ax=axes[i], color='red', linewidth=2)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Histograma + densidad superpuesta (normalizado)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Histogramas con diferentes números de bins (Sturges, sqrt, Freeman-Diaconis)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
bins_methods = [('Sturges', 30), ('sqrt(n)', int(np.sqrt(len(data)))), ('50 bins', 50)]
for k, (label, n_bins) in enumerate(bins_methods):
    n_bins = int(n_bins) if isinstance(n_bins, float) else n_bins
    data[numeric_cols[0]].dropna().hist(bins=n_bins, ax=axes[k], edgecolor='black', alpha=0.7)
    axes[k].set_title(f'{numeric_cols[0]} - {label}')
plt.suptitle('Comparación de histogramas con distintos números de bins', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Histogramas horizontales (stacked) - todas las variables en escala comparable
data_norm = data[numeric_cols].dropna()
data_norm = (data_norm - data_norm.min()) / (data_norm.max() - data_norm.min() + 1e-8)
fig, ax = plt.subplots(figsize=(10, 6))
for col in numeric_cols:
    ax.hist(data_norm[col], bins=25, alpha=0.4, label=col[:15], density=True)
ax.legend(bbox_to_anchor=(1.02, 1), fontsize=8)
ax.set_title('Distribuciones superpuestas (datos normalizados 0-1)')
ax.set_xlabel('Valor normalizado')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Box plots (diagramas de caja) - vista de dispersión y cuartiles
fig, ax = plt.subplots(figsize=(12, 5))
data[numeric_cols].boxplot(ax=ax, rot=45)
ax.set_title('Diagramas de caja de las variables')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Gráficos de violín (violin plots) - densidad + box
fig, ax = plt.subplots(figsize=(12, 5))
data_melt = data[numeric_cols].melt(var_name='Variable', value_name='Valor')
sns.violinplot(data=data_melt, x='Variable', y='Valor', ax=ax)
ax.tick_params(axis='x', rotation=45)
ax.set_title('Gráficos de violín: distribución de cada variable')
plt.tight_layout()
plt.show()

In [ ]:
# 8. Seaborn histplot con KDE integrado
n_vars = len(numeric_cols)
n_rows = (n_vars + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(14, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(data[col].dropna(), kde=True, bins=25, ax=axes[i])
    axes[i].set_title(col, fontsize=9)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Histogramas con KDE (Seaborn)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 9. Histogramas acumulados (CDF) - función de distribución empírica
n_vars = len(numeric_cols)
n_rows = (n_vars + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(14, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()
for i, col in enumerate(numeric_cols):
    sorted_vals = np.sort(data[col].dropna())
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    axes[i].plot(sorted_vals, cdf)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_ylabel('F(x)')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Funciones de distribución acumulada (CDF empírica)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación (heatmap)
corr_matrix = data[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Matriz de correlación entre variables')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Estimacion Modelos Regularizados

Desde la librería sklearn importaremos todos los modelos que utilizaremos en esta sesión, así como las métricas necesarias.

In [ ]:
''' MODELOS a Estimar '''
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
'''METRICS'''
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error



## Separando Muestras (Training v/s Testing)


Como es usual, definiendo una fecha de corte. Separaremos la data en dos muestras. Una de entrenamiento (training) y otra para evaluar el desempeño de dichos modelos en una data diferente a la que fueron entrenados (testing)

In [ ]:
# Separando variable explicada (y) de explicativas (x)
# Configurar la columna de fecha como índice para operaciones de series temporales
data_processed = data.set_index(data.columns[0])
data_processed.index = pd.to_datetime(data_processed.index)

# Definir y (objetivo) y x (predictores): USDCLP Curncy es la variable objetivo
y = data_processed.iloc[:, 0]
# Las características son las columnas restantes, excluyendo la última (1:-1)
x = data_processed.iloc[:, 1:-1]

# Umbral de corte para división entrenamiento/prueba
year_corte = '2016-06-19 00:00:00'
year_corte_dt = pd.to_datetime(year_corte)

# Entrenamiento (antes de la fecha de corte)
x_tr = x[x.index < year_corte_dt].copy(True)
y_tr = y[y.index < year_corte_dt].copy(True)
# Prueba (desde la fecha de corte, >= para incluir ese día)
x_tst = x[x.index >= year_corte_dt].copy(True)
y_tst = y[y.index >= year_corte_dt].copy(True)

## Modelo de Regresión Lineal: Análisis y Diagnósticos

Antes de comparar con Ridge, Lasso y Elastic Net, evaluamos un modelo de regresión lineal con las variables originales:
1. **Forward selection**: R2, AIC, BIC
2. **Significancia de los betas** (p-valores)
3. **Normalidad de residuos** (Shapiro-Wilk, Q-Q)
4. **Autocorrelación** (Durbin-Watson, Ljung-Box)
5. **Heterocedasticidad** (Breusch-Pagan, White)

In [ ]:
try:
    import statsmodels.api as sm
    from statsmodels.stats.diagnostic import het_breuschpagan, het_white, acorr_ljungbox
    from statsmodels.stats.stattools import durbin_watson
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'statsmodels', '-q'])
    import statsmodels.api as sm
    from statsmodels.stats.diagnostic import het_breuschpagan, het_white, acorr_ljungbox
    from statsmodels.stats.stattools import durbin_watson
from scipy.stats import shapiro
import numpy as np

# Forward selection: R2, AIC, BIC
X_tr_const = sm.add_constant(x_tr)
remaining = list(x_tr.columns)
selected = []
results_fwd = []

while remaining:
    best_aic, best_var = np.inf, None
    for var in remaining:
        cols = ['const'] + selected + [var]
        model = sm.OLS(y_tr, X_tr_const[cols]).fit()
        if model.aic < best_aic:
            best_aic, best_bic, best_r2 = model.aic, model.bic, model.rsquared
            best_var = var
    if best_var is None:
        break
    selected.append(best_var)
    remaining.remove(best_var)
    results_fwd.append({'paso': len(selected), 'variable': best_var, 'R2': best_r2, 'AIC': best_aic, 'BIC': best_bic})

print("Forward Selection - R2, AIC, BIC por paso:")
display(pd.DataFrame(results_fwd))

### Modelo OLS final (todas las variables seleccionadas por forward)

In [ ]:
cols_final = ['const'] + selected
model_ols = sm.OLS(y_tr, X_tr_const[cols_final]).fit()
print("Resumen del modelo OLS - Significancia de betas (p-valores):")
print(model_ols.summary())

### 1. Normalidad de residuos

In [ ]:
# 1. Normalidad de residuos (Shapiro-Wilk, Q-Q plot)
residuos = model_ols.resid
stat_shapiro, p_shapiro = shapiro(residuos)
print(f"Shapiro-Wilk: estadístico={stat_shapiro:.4f}, p-valor={p_shapiro:.4f}")
print("H0: residuos normales. p<0.05 → rechazar normalidad")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuos, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Histograma de residuos')
axes[0].set_xlabel('Residuos')
sm.qqplot(residuos, line='45', ax=axes[1])
axes[1].set_title('Q-Q plot (normalidad)')
plt.tight_layout()
plt.show()

### 2. Autocorrelación (Durbin-Watson, Ljung-Box)

In [ ]:
dw = durbin_watson(residuos)
print(f"Durbin-Watson: {dw:.4f} (≈2: no autocorr; <1.5 o >2.5: posible autocorr)")

lb_test = acorr_ljungbox(residuos, lags=10, return_df=True)
print("\nLjung-Box (H0: no autocorr hasta lag k):")
print(lb_test)

### 3. Heterocedasticidad (Breusch-Pagan, White)

In [ ]:
exog = X_tr_const[cols_final]
bp_lm, bp_pval, bp_fstat, bp_fpval = het_breuschpagan(residuos, exog)
print(f"Breusch-Pagan: LM={bp_lm:.4f}, p-valor={bp_pval:.4f}")
print("H0: homocedasticidad. p<0.05 → heterocedasticidad")

white_lm, white_pval, white_fstat, white_fpval = het_white(residuos, exog)
print(f"\nWhite: LM={white_lm:.4f}, p-valor={white_pval:.4f}")

# Funcion Generadora de Rezagos

Esta función nos permitirá generar rezagos de las variables disponibles en la base de datos.


In [ ]:
def crealags(base,lag_ini,nrolags):
    for lags in range(lag_ini,  nrolags+lag_ini): #Parte del rezago que definamos, no desde 1
        slag=base.shift(lags).copy(True)
        slag.columns=[str(col) + '_lag'+str(lags) for col in base.columns]
        if lags==lag_ini:#Bloque de datos inicial
            rezagos=pd.concat([slag], axis=1).copy(True)#genera primer bloque de datos
        else: rezagos=pd.concat([rezagos,slag], axis=1).copy(True) #genera el resto del bloque de datos
    return rezagos

## Generando Rezagos en la Data

Haciendo uso de la función recién generada, generaremos rezagos para cada una de las variables del dataframe que hemos importado desde excel. Para cada variable (incluye la dependiente), se han generado 5 rezagos.

In [ ]:
nrolags=5
lag_ini = 1
rezagos = crealags(data,lag_ini,nrolags)
rezagos.head(10)

## Muestras training y testing con rezagos

In [ ]:
# First, ensure rezagos has a datetime index and drop redundant lagged date columns
# rezagos comes from crealags(data, lag_ini, nrolags), which used the raw `data` (integer index).
# So, `rezagos` itself has an integer index.
# We need to make it datetime indexed for proper alignment and slicing.
rezagos_indexed = rezagos.copy()
# Find the column that contains the original date (e.g., 'Unnamed: 0_lag1') to set as index
date_lag_col = [col for col in rezagos_indexed.columns if 'Unnamed: 0_lag' in col][0]
rezagos_indexed = rezagos_indexed.set_index(date_lag_col)
rezagos_indexed.index = pd.to_datetime(rezagos_indexed.index)
# Drop all 'Unnamed: 0_lagX' columns from the data, as the index now handles the date
rezagos_indexed = rezagos_indexed.drop(columns=[col for col in rezagos_indexed.columns if 'Unnamed: 0_lag' in col])

# y_original is the non-lagged target variable, which is `USDCLP Curncy` from `data_processed`
# (data_processed already has a datetime index)
y_original = data_processed['USDCLP Curncy']

# Concatenate the original target with the lagged features.
# This creates a DataFrame where the first column is the original target,
# and subsequent columns are all the lagged features, correctly aligned by date.
data_rezagos = pd.concat([y_original, rezagos_indexed], axis=1).dropna()

# Now define y (target) and x (features) from data_rezagos
y = data_rezagos.iloc[:, 0]
x = data_rezagos.iloc[:, 1:] # All other columns are lagged features

# Umbral de Corte
year_corte = '2016-06-19 00:00:00'
# Convert year_corte to a datetime object for proper comparison with the DatetimeIndex
year_corte_dt = pd.to_datetime(year_corte)

# Trainig
x_tr = x[x.index < year_corte_dt].copy(True)
y_tr = y[y.index < year_corte_dt].copy(True)
# Testing
x_tst = x[x.index >= year_corte_dt].copy(True) # Using >= to include the cutoff date in the testing set
y_tst = y[y.index >= year_corte_dt].copy(True)

## Concatenando Rezagos con Data Original

Lo que ahora haremos será concatenar ambos DataFrame, eliminando en el proceso, todos los datos que perdimos al realizar los rezagos de las variables (**.dropna()**)

In [ ]:
data_rezagos=pd.concat([data.iloc[:,0],rezagos],axis=1).dropna()

In [ ]:
data_rezagos

## Declaracion de Modelos a Estimar (forma conveniente)

En esta función, declararemos todos los modelos a utilizar, dentro de listas que ocuparemos más adelante.

In [ ]:
def creamodelos():
    print('Creando Modelos...')
    names = [
        "LinearRegression",
        'Ridge',
        'Lasso',
        'ElasticNet'
    ]

    regresores = [
        LinearRegression(),
        Ridge(),
        Lasso(),
        ElasticNet(),
    ]

    print('Lista de modelos creados...')
    print(names)
    print('\n')
    return names, regresores

names, regresores = creamodelos()

# Estimación + Métricas Training

Para ajustar los modelos que hemos declarado anteriormente (**creamodelos**), haremos uso de un loop que irá tomando cada uno de estos modelos declarados, estimándolo y simultáneamente calculando sus métricas de desempeño en un DataFrame que consolida toda la información.

In [ ]:
modelos=[]
metricas_training=[]
counter=1
for name, modelo in zip(names, regresores):
    #print("entrenando modelo")
    #print(name)

    modelo.fit(x_tr,y_tr)

    #guarda el objeto modelo entrenado y colocale un id counter
    modelos.append({'algo':modelo,'idmod':counter})
    pred_training=modelo.predict(x_tr)

    def metricas(y_real,pred,counter,name):
        res=[]
        #%%Creacion y exportacion de métricas del modelo.
        MAE=mean_absolute_error(y_real,pred)
        RMSE=(mean_squared_error(y_real,pred))**0.5

        res.append((counter,name,MAE,RMSE))
        return res

    res_tr=metricas(y_tr,pred_training,counter,name)
    metricas_training.append(res_tr)
    counter=counter+1


from itertools import chain
def flatten(listOfLists):
    "Flatten one level of nesting"
    return chain.from_iterable(listOfLists)

metricas_training=list(flatten(metricas_training))
resultados_training=pd.DataFrame(data=metricas_training, columns=['id_modelo','Modelo','MAE','RMSE'])
resultados_training

# Estimacion + Métricas Testing

Repetimos el mismo proceso para la muestra de testing.

In [ ]:
# IMPORTANTE: Usamos los modelos YA ENTRENADOS en training (celda anterior).
# NO re-entrenamos en testing; eso sería un error metodológico (data leakage).
metricas_testing=[]
counter=1
for name, mod_dict in zip(names, modelos):
    modelo = mod_dict['algo']  # modelo ya entrenado en x_tr, y_tr
    pred_testing=modelo.predict(x_tst)

    def metricas(y_real,pred,counter,name):
        res=[]
        #%%Creacion y exportacion de métricas del modelo.
        MAE=mean_absolute_error(y_real,pred)
        RMSE=(mean_squared_error(y_real,pred))**0.5

        res.append((counter,name,MAE,RMSE))
        return res

    res_tst=metricas(y_tst,pred_testing,counter,name)
    metricas_testing.append(res_tst)
    counter=counter+1



metricas_testing=list(flatten(metricas_testing))
resultados_testing=pd.DataFrame(data=metricas_testing, columns=['id_modelo','Modelo','MAE','RMSE'])
resultados_testing
